In [6]:
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
import ollama
from openai import OpenAI

In [4]:
#using ollama -- my adition--
#
from openai import OpenAI
ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')   
MODEL_llama = "llama3.2"
MODEL_deep = "deepseek-v2"

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website_func:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

system_prompt = "You are an assistant who provides a short summary of a website passed to you\
respond with key points"

def user_prompt(website):
    new_site = Website_func(website).text
    user_prompt = "Please provide a short summary for the website, respond with key points\n"
    user_prompt += new_site
    return user_prompt

def message_for(website):
    return [
        {"role":"system", "content":system_prompt},
        {"role":"user","content":user_prompt(website)}
         ]
    
def summerizer(website):
    
    response = ollama.chat(model=MODEL_llama, messages=message_for(website))
    return(response)
    print(response['message']['content'])
    
    '''
    uses openai with llama3.2 model
    response = ollama_via_openai.chat.completions.create(
                model = MODEL_llama,
                messages = message_for(website),
                stream = False
    )
    return response.choices[0].message.content
  '''
    
def display_summery(website):
    summary = summerizer(website)
    display(Markdown(summary))

    

In [8]:
#ollama
url = "https://edwarddonner.com"
summerizer(url)

ChatResponse(model='llama3.2', created_at='2025-02-07T23:50:41.975824Z', done=True, done_reason='stop', total_duration=13165961096, load_duration=27027041, prompt_eval_count=434, prompt_eval_duration=82000000, eval_count=156, eval_duration=13054000000, message=Message(role='assistant', content='Here are the key points from the website:\n\n* **Nebula.io**: AI-powered platform that helps people discover their potential and pursue their reason for being.\n* **LLM Arena**: "Outsmart" arena where LLMs compete against each other in a battle of diplomacy and deviousness.\n* **Ed (Founder & CTO)**: Co-founder and CTO of Nebula.io, previously founder and CEO of AI startup untapt (acquired in 2021).\n* **Expertise**: Expertise in applying AI to talent management and development.\n* **Proprietary LLMs**: Nebula.io uses groundbreaking, proprietary LLMs verticalized for talent.\n* **Awards and Recognition**: Award-winning platform with happy customers and press coverage.', images=None, tool_calls=N